extract every hashtag from the v3 advocacy corpus, count occurrences, and export anything appearing at least 10 times. broken down by band (from nb 05) so the election-themed hashtags can be lifted into the seed list for nb 06.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lower, regexp_extract_all, explode, concat_ws, array_join,
    count as spark_count, sum as spark_sum, desc, broadcast, lit,
)
import pandas as pd

spark = SparkSession.builder \
    .appName('FB_API_hashtag_analysis') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

In [ ]:
V3_PATH    = '/user/s3348393/main/preprocessing/v3/parquet'
BANDS_CSV  = '../data/all_bylines_bands.csv'
OUTPUT_CSV = '../data/hashtags.csv'
MIN_COUNT  = 10

load v3 advocacy ads and attach band labels.

In [ ]:
v3 = spark.read.parquet(V3_PATH).filter(col('match_type').isNull())

# combine body + title arrays into one text blob per ad. don't lowercase yet —
# we want to extract hashtags first, then lowercase them so #AusVotes and
# #ausvotes collapse to the same row.
v3 = v3.withColumn('text', concat_ws(' ',
    array_join('creative_bodies', ' '),
    array_join('creative_link_titles', ' ')))

v3 = v3.filter(col('text').isNotNull() & (col('text') != ''))

# attach band labels (left join — bylines outside the qualifying set get null band).
try:
    bands_pdf = pd.read_csv(BANDS_CSV)[['bylines', 'band']]
    bands_sdf = spark.createDataFrame(bands_pdf)
    v3 = v3.join(broadcast(bands_sdf), 'bylines', 'left')
    have_bands = True
    print(f'Loaded {len(bands_pdf)} band-classified bylines.')
except FileNotFoundError:
    v3 = v3.withColumn('band', lit(None).cast('string'))
    have_bands = False
    print(f'{BANDS_CSV} not found — running without band breakdown. '
          'run nb 05 to generate the bands csv.')

v3 = v3.cache()
print(f'Advocacy ads with text: {v3.count():,}')

extract hashtags with a regex. regexp_extract_all returns every match per row as an array, which we explode to one hashtag per row. lowercase after extraction so capitalisation variants collapse.

In [ ]:
# matches #word — letters, digits and underscores after the hash, at least 2 chars.
HASHTAG_RE = r'#([A-Za-z0-9_]{2,})'

hashtag_rows = (v3
    .withColumn('hashtags', regexp_extract_all(col('text'), lit(HASHTAG_RE), 1))
    .filter(col('hashtags').isNotNull())
    .select('bylines', 'band', explode(col('hashtags')).alias('hashtag'))
    .withColumn('hashtag', lower(col('hashtag'))))

hashtag_rows = hashtag_rows.cache()
print(f'Total hashtag occurrences: {hashtag_rows.count():,}')
print(f'Distinct hashtags:         {hashtag_rows.select("hashtag").distinct().count():,}')

count by hashtag overall, then pivot in counts per band so election-themed hashtags are visually obvious. filter to hashtags occurring at least MIN_COUNT times.

In [ ]:
overall = (hashtag_rows.groupBy('hashtag').count()
    .withColumnRenamed('count', 'count_total')
    .filter(col('count_total') >= MIN_COUNT))

if have_bands:
    per_band = (hashtag_rows
        .groupBy('hashtag', 'band').count()
        .groupBy('hashtag')
        .pivot('band')
        .sum('count')
        .na.fill(0))
    out = overall.join(per_band, 'hashtag', 'left').orderBy(desc('count_total'))
else:
    out = overall.orderBy(desc('count_total'))

out_pdf = out.toPandas()

# tidy column order — overall count first, then per-band counts (if present).
band_cols = [c for c in ['election_only', 'transitional', 'ongoing', 'off_campaign']
             if c in out_pdf.columns]
out_pdf = out_pdf[['hashtag', 'count_total'] + band_cols +
                   [c for c in out_pdf.columns if c not in ['hashtag', 'count_total'] + band_cols]]

print(f'Hashtags with count >= {MIN_COUNT}: {len(out_pdf):,}')
out_pdf.head(40)

write the csv. one row per hashtag, sorted by total count descending.

In [ ]:
out_pdf.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {OUTPUT_CSV} ({len(out_pdf):,} hashtags)')

top hashtags in the election_only band only — these are the candidates for the seed list in nb 06.

In [ ]:
if have_bands and 'election_only' in out_pdf.columns:
    election_top = (out_pdf[out_pdf['election_only'] >= MIN_COUNT]
                    .sort_values('election_only', ascending=False)
                    .head(40)
                    .reset_index(drop=True))
    display(election_top[['hashtag', 'election_only', 'count_total']])
else:
    print('Run nb 05 to generate band labels, then re-run this cell.')